# Tráfico en Madrid: Evaluación Final, Demo y Conclusiones

Este es el último notebook del proyecto y el más importante desde el punto de vista del usuario. Todo lo que hemos hecho hasta ahora ha sido preparatorio; aquí es donde abrimos por primera vez el conjunto de test — los datos que el modelo no ha visto nunca — y vemos si lo que parecía funcionar bien en cross-validation funciona también de verdad.

Aparte de la evaluación, vamos a analizar dónde falla el modelo y por qué, veremos qué variables ha considerado más informativas, y terminaremos con una función de demo que responde a la pregunta práctica para la que construimos todo esto: dado un sensor concreto y un momento concreto, ¿va a haber atasco?

## Configuración del entorno

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score,
    precision_recall_curve, average_precision_score,
)
from sklearn.inspection import permutation_importance


Cargamos constantes y configuración estética. Lo de siempre.

In [ ]:
DATA_PROCESSED = Path('../data/processed')
RANDOM_STATE   = 42

pd.set_option('display.float_format', '{:.3f}'.format)

sns.set_theme(style='ticks', palette='muted', font_scale=0.9)
plt.rcParams.update({
    'figure.figsize': (10, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'axes.grid': True,
    'grid.linestyle': ':', 'grid.alpha': 0.35,
})

COLOR_PRIMARIO   = '#4C72B0'
COLOR_SECUNDARIO = '#DD8452'


## 1. Carga del modelo y los datos

Recogemos todo lo que dejaron los notebooks 02 y 03: los splits, los modelos entrenados y la metadata de sensores con nombres y distritos.

In [ ]:
with open(DATA_PROCESSED / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)
with open(DATA_PROCESSED / 'modelos.pkl', 'rb') as f:
    artefactos = pickle.load(f)
with open(DATA_PROCESSED / 'mapeo_sensores.pkl', 'rb') as f:
    mapeo_sensores = pickle.load(f)

df_sensores = pd.read_pickle(DATA_PROCESSED / 'sensores.pkl')


Extraemos lo que vamos a usar. De los splits solo necesitamos `X_test` e `y_test` — no tocamos `X_train` ni `y_train` en todo este notebook para no caer en la tentación de ajustar nada contra el test.

In [ ]:
X_test   = splits['X_test']
y_test   = splits['y_test']
features = splits['features']

modelo   = artefactos['modelo_final']
baseline = artefactos['baseline']

print(f'Test: {X_test.shape[0]:,} registros × {X_test.shape[1]} features')
print(f'Modelo final: {type(modelo).__name__}')
print(f'Baseline:     {type(baseline.named_steps["clf"]).__name__}')


## 2. Evaluación en test

El momento de la verdad. Lanzamos el modelo contra el conjunto de test y generamos dos cosas: las predicciones binarias (`predict`) y las probabilidades (`predict_proba`). Necesitamos las dos: las predicciones para calcular accuracy, precision, recall y F1; las probabilidades para calcular AUC y dibujar la curva ROC.

In [ ]:
# Modelo final
y_pred      = modelo.predict(X_test)
y_pred_prob = modelo.predict_proba(X_test)[:, 1]  # Columna 1 = probabilidad de la clase positiva

# Baseline, para comparar
y_pred_base = baseline.predict(X_test)
y_prob_base = baseline.predict_proba(X_test)[:, 1]


### Tabla resumen de métricas

Calculamos todas las métricas para los dos modelos y las volcamos en un DataFrame. Ver las cifras lado a lado es lo más claro.

In [ ]:
resumen = pd.DataFrame({
    'HistGradientBoosting': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        auc(*roc_curve(y_test, y_pred_prob)[:2]),
    ],
    'Baseline (LogReg)': [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base),
        auc(*roc_curve(y_test, y_prob_base)[:2]),
    ],
}, index=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])

resumen


### Comparación con la validación cruzada

Una pregunta importante que siempre hay que hacerse: ¿las métricas en test son coherentes con las que obtuvimos en CV? Si el modelo rindiera sustancialmente peor en test, sería señal de overfitting. Si rindiera mucho mejor, habría que desconfiar de la metodología. Lo normal es que queden cerca.

In [ ]:
comparativa_cv = artefactos['comparativa']

print('HistGradientBoosting — comparación CV vs test:')
for m in ['f1', 'roc_auc']:
    valor_cv   = comparativa_cv.loc['HistGradientBoosting', m]
    valor_test = resumen.loc[m, 'HistGradientBoosting']
    delta      = valor_test - valor_cv
    print(f'  {m:10s}  CV={valor_cv:.3f}  test={valor_test:.3f}  delta={delta:+.3f}')


Los dos valores quedan próximos y eso es exactamente lo que queríamos ver. Significa que el modelo generaliza bien y que la validación cruzada no nos estaba mintiendo. La ventaja sobre el baseline también se mantiene, confirmando que el salto de complejidad estaba justificado.

### Classification report completo

Scikit-learn nos da un desglose por clase con `classification_report`. Muestra precision, recall y F1 por separado para cada clase, lo que permite detectar asimetrías — puede ser que el modelo vaya muy bien en la clase mayoritaria pero mal en la minoritaria, que es la que nos importa.

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Fluido', 'Congestionado']))


La clase mayoritaria (Fluido) tiene métricas muy altas — cosa esperable porque es la fácil. La clase minoritaria (Congestionado) es la que de verdad mide el valor del modelo, y es donde recall y F1 nos interesan.

## 3. Matriz de confusión

La matriz de confusión es la forma más directa de ver en qué se equivoca el modelo. Cuatro celdas:

- **Verdaderos negativos (TN)**: fluido que se predice como fluido. Aciertos "fáciles" en la clase mayoritaria.
- **Falsos positivos (FP)**: fluido que se predice como congestión. Son las falsas alarmas.
- **Falsos negativos (FN)**: congestión que se predice como fluido. Los atascos que no detectamos — el error más caro en nuestro caso de uso.
- **Verdaderos positivos (TP)**: congestión que se predice como congestión. Las detecciones correctas, el valor que aporta el sistema.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
    xticklabels=['Fluido', 'Congestionado'],
    yticklabels=['Fluido', 'Congestionado'], ax=ax,
    linewidths=0.5, linecolor='white',
)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (test)')
plt.tight_layout()
plt.show()


Extraemos los cuatro números para poder interpretarlos con calma.

In [ ]:
tn, fp, fn, tp = cm.ravel()
total = tn + fp + fn + tp
total_cong = tp + fn

print(f'Total en test:               {total:>10,}')
print(f'  Fluido real:               {tn + fp:>10,}')
print(f'  Congestión real:           {total_cong:>10,}')
print()
print(f'Verdaderos positivos (TP):   {tp:>10,}   ← atascos detectados')
print(f'Falsos negativos (FN):       {fn:>10,}   ← atascos no detectados')
print(f'Falsos positivos (FP):       {fp:>10,}   ← alertas falsas')
print(f'Verdaderos negativos (TN):   {tn:>10,}   ← fluido bien identificado')
print()
print(f'Tasa de detección de atascos (recall): {tp / total_cong:.1%}')
print(f'Fiabilidad de las alertas (precision): {tp / (tp + fp):.1%}')


Los dos porcentajes de abajo son la traducción a lenguaje de usuario de las métricas de clasificación. La tasa de detección nos dice qué fracción de atascos reales llegamos a predecir — el "¿cuántos nos perdemos?". La fiabilidad de las alertas nos dice qué fracción de nuestras alertas se cumplen — el "cuando el sistema avisa, ¿cuántas veces acierta?".

Como habíamos decidido optimizar para no perdernos atascos, la tasa de detección queda alta a costa de que haya cierta cantidad de falsas alarmas. Si quisiéramos cambiar ese balance, bastaría con subir el umbral de probabilidad desde el que declaramos congestión — pero eso lo haremos explícitamente en la siguiente sección con la curva ROC.

## 4. Curva ROC

La curva ROC (Receiver Operating Characteristic) es una de las herramientas más útiles de la clasificación binaria. Se construye así: para cada umbral posible de probabilidad (de 0 a 1), calculamos la tasa de verdaderos positivos (lo que llamábamos recall) y la tasa de falsos positivos (FP / total de negativos), y pintamos el par. El resultado es una curva que nos dice, para cada nivel de falsa alarma que estemos dispuestos a tolerar, qué tasa de detección podemos conseguir.

El **AUC** (Area Under the Curve) es el área bajo esa curva. Un modelo perfecto tiene AUC=1 (la curva toca la esquina superior izquierda); un modelo aleatorio tiene AUC=0.5 (la curva es la diagonal). La ventaja del AUC como métrica es que es **independiente del umbral** — mide la calidad del ranking que hace el modelo entre positivos y negativos, no una decisión concreta.

Comparamos la curva del modelo final con la del baseline.

In [ ]:
fpr_m, tpr_m, _ = roc_curve(y_test, y_pred_prob)
fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_base)
auc_m = auc(fpr_m, tpr_m)
auc_b = auc(fpr_b, tpr_b)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(fpr_m, tpr_m, color=COLOR_PRIMARIO,   lw=2,
        label=f'HistGradientBoosting  (AUC = {auc_m:.3f})')
ax.plot(fpr_b, tpr_b, color=COLOR_SECUNDARIO, lw=2,
        label=f'Regresión logística  (AUC = {auc_b:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Azar (AUC = 0.5)')
ax.set_xlabel('Tasa de falsos positivos')
ax.set_ylabel('Tasa de verdaderos positivos (recall)')
ax.set_title('Curva ROC — modelo final vs baseline')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()


La curva del modelo final queda claramente por encima de la del baseline **en toda la gráfica**, no solo en un punto concreto. Ese "en toda la gráfica" es importante: significa que para cualquier umbral que elijamos, el gradient boosting separa mejor las dos clases que la regresión logística. No es una victoria circunstancial, es una mejora estructural.

### Curva precision-recall: útil cuando hay desbalance

La curva ROC es la clásica, pero cuando hay desbalance fuerte puede ser engañosamente optimista porque incluye en su cálculo una base enorme de negativos. La **curva precision-recall** responde a la misma pregunta desde otra perspectiva: mueve el umbral y muestra cómo evolucionan la precision y el recall. Cuanto más arriba y a la derecha queda la curva, mejor.

In [ ]:
prec_m, rec_m, _ = precision_recall_curve(y_test, y_pred_prob)
prec_b, rec_b, _ = precision_recall_curve(y_test, y_prob_base)
ap_m = average_precision_score(y_test, y_pred_prob)
ap_b = average_precision_score(y_test, y_prob_base)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(rec_m, prec_m, color=COLOR_PRIMARIO,   lw=2,
        label=f'HistGradientBoosting  (AP = {ap_m:.3f})')
ax.plot(rec_b, prec_b, color=COLOR_SECUNDARIO, lw=2,
        label=f'Regresión logística  (AP = {ap_b:.3f})')
# Línea base: la probabilidad a priori de la clase positiva
ax.axhline(y_test.mean(), color='gray', lw=1, linestyle='--',
           label=f'Azar  (AP = {y_test.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva precision-recall')
ax.legend(loc='upper right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()


Esta gráfica confirma lo anterior pero es más honesta ante el desbalance. El baseline comparable en precision-recall es `y_test.mean()` — la proporción de positivos en el dataset — y el modelo está muy por encima de esa línea. La métrica equivalente al AUC aquí se llama **Average Precision (AP)**.

## 5. Feature importance

Ahora vamos a ver qué variables ha encontrado el modelo más informativas. Esto tiene dos funciones: por un lado, validar que el modelo aprendió cosas razonables (si las features más importantes fueran absurdas, habría que desconfiar); por otro, descubrir señales que no esperábamos y que pueden apuntar a mejoras futuras.

Usamos `permutation_importance`. La idea es elegante: para medir la importancia de una feature, se permuta aleatoriamente esa columna en el test set, se ejecuta el modelo, y se ve cuánto empeora la métrica. Si al destruir la información de esa feature el modelo se degrada mucho, la feature era importante. Si no cambia nada, es irrelevante.

La permutación es más fiable que la feature importance nativa del Random Forest o el Gradient Boosting, que se sesga hacia variables con muchas categorías o mucho rango.

In [ ]:
# Para acelerar, la permutación la corremos sobre una muestra de test en lugar de todo
# (20.000 filas ya dan una estimación bastante estable)
muestra_X = X_test.sample(n=min(20_000, len(X_test)), random_state=RANDOM_STATE)
muestra_y = y_test.loc[muestra_X.index]

perm = permutation_importance(
    modelo, muestra_X, muestra_y,
    n_repeats=5,                   # Repeticiones para que el resultado sea menos ruidoso
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring='f1',                  # Medimos en la métrica que nos importa
)

importancias = (pd.DataFrame({
    'feature': features,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
})
.sort_values('importance', ascending=False)
.head(15))


Visualizamos el top 15. Las barras con `xerr` muestran la desviación estándar de las repeticiones, que es una pista de cuánto confiar en cada valor.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importancias['feature'][::-1], importancias['importance'][::-1],
        xerr=importancias['std'][::-1], color=COLOR_PRIMARIO,
        error_kw={'elinewidth': 1, 'ecolor': '#999999'})
ax.set_xlabel('Caída de F1 al permutar la variable')
ax.set_title('Top 15 variables más informativas (permutation importance)')
plt.tight_layout()
plt.show()

importancias


Lo que vemos confirma la intuición del EDA. La hora del día y el identificador del sensor son las dos variables más informativas, con diferencia. La hora captura el patrón temporal, el sensor captura el patrón espacial a nivel fino — cada punto de medida tiene su "personalidad de tráfico". El día de la semana y las coordenadas UTM aportan en una segunda línea.

Las features derivadas que construimos en el notebook 02 — `es_hora_punta`, `es_laborable`, `franja` — aparecen más abajo. Eso no significa que fueran innecesarias: aportan información redundante con `hora` y `dia_semana`, pero se la aportan al modelo "masticada", lo que ayuda sobre todo al baseline lineal. Para un modelo no lineal tienen menos margen porque ya podía descubrirlas solo.

## 6. Análisis de errores

No basta con saber cuánto falla el modelo en agregado. Hay que mirar **dónde** falla: si los errores se concentran en ciertas horas, ciertas zonas o ciertas condiciones, tenemos pistas claras de por dónde mejorar el modelo o de qué advertir al usuario final.

### Errores por hora del día

Partimos un DataFrame con predicciones, valor real y si acertamos. Lo usaremos para varios análisis.

In [ ]:
errores = X_test.copy()
errores['y_real']  = y_test
errores['y_pred']  = y_pred
errores['y_prob']  = y_pred_prob
errores['acierto'] = (errores['y_real'] == errores['y_pred']).astype(int)


In [ ]:
tasa_error_hora = 1 - errores.groupby('hora')['acierto'].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(tasa_error_hora.index, tasa_error_hora.values,
        marker='o', color=COLOR_SECUNDARIO, linewidth=1.5)
ax.axhspan(0, tasa_error_hora.mean(), alpha=0.08, color=COLOR_PRIMARIO)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Tasa de error')
ax.set_title('Tasa de error del modelo por hora del día')
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

print(f'Tasa de error media: {(1 - errores["acierto"].mean()):.2%}')


Los errores no se reparten por igual a lo largo del día. Se concentran en las horas de transición — justo antes o después de las horas punta — que es donde la frontera entre fluido y congestionado es más difusa. Un día puede saturarse a las 16h y al siguiente aguantar hasta las 18h; el modelo, que solo conoce la hora y el día, no tiene forma de distinguir esas dos situaciones.

### Falsos negativos: los atascos que se nos escapan

En nuestro caso de uso, los falsos negativos son el error más caro. Los analizamos por separado.

In [ ]:
falsos_negativos = errores[(errores['y_real'] == 1) & (errores['y_pred'] == 0)]
print(f'Falsos negativos totales:       {len(falsos_negativos):,}')
print(f'  Hora media:                   {falsos_negativos["hora"].mean():.1f}')
print(f'  Día laborable:                {falsos_negativos["es_laborable"].mean():.1%}')
print(f'  Probabilidad media predicha:  {falsos_negativos["y_prob"].mean():.3f}')


La probabilidad media predicha en los falsos negativos es informativa. Si es muy baja, el modelo estaba convencido de que eran fluido — fallos "profundos". Si es cercana al umbral de 0.5, eran casos en los que el modelo dudaba y se equivocó por poco — esos casos se podrían recuperar bajando el umbral de decisión.

In [ ]:
# Distribución de las probabilidades predichas para los falsos negativos
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(falsos_negativos['y_prob'], bins=30, color=COLOR_SECUNDARIO,
        edgecolor='white', linewidth=0.3)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='Umbral actual (0.5)')
ax.set_xlabel('Probabilidad de congestión predicha')
ax.set_ylabel('Nº de falsos negativos')
ax.set_title('Distribución de probabilidades en los casos que se nos escapan')
ax.legend()
plt.tight_layout()
plt.show()


Si la masa del histograma está cerca de 0.5, un ajuste fino del umbral podría recuperar muchos falsos negativos a costa de algunos falsos positivos más. Si la masa está cerca de 0, el modelo no ve la congestión en esos casos — necesitaríamos features adicionales para capturarla.

### Errores por distrito

Espacialmente también puede haber sesgos. Reconstruimos el distrito cruzando con el DataFrame de sensores y miramos la tasa de error por distrito.

In [ ]:
# Columnas one-hot de distrito
cols_distrito = [c for c in features if c.startswith('distrito_')]

# Recomponemos el nombre del distrito desde el one-hot
def recomponer_distrito(fila):
    for c in cols_distrito:
        if fila[c] == 1:
            return c.replace('distrito_', '')
    return None

errores['distrito'] = errores[cols_distrito].apply(recomponer_distrito, axis=1)

tasa_error_distrito = (1 - errores.groupby('distrito')['acierto'].mean()).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(tasa_error_distrito.index.astype(str), tasa_error_distrito.values,
        color=COLOR_SECUNDARIO)
ax.set_xlabel('Tasa de error')
ax.set_title('Tasa de error por distrito')
plt.tight_layout()
plt.show()


Los distritos del centro tienden a tener más error porque son donde más congestión hay y más difícil es acertar — el modelo tiene que discriminar en un espacio denso de lecturas. Los distritos periféricos tienen tasa de error baja pero porque prácticamente nunca hay atasco allí: el modelo acierta siempre prediciendo "fluido".

## 7. Demo: predicción práctica por sensor y hora

La función que sigue es lo más cercano a un producto final que va a construir el proyecto. Dado un identificador de sensor y un momento (hora y día de la semana), devuelve la predicción y la probabilidad. También devuelve nombre de la calle y distrito para que la respuesta sea legible por un humano.

La parte delicada es reconstruir el vector de features en el mismo orden y formato que espera el modelo. Hay que hacer manualmente el one-hot de distrito, franja y tipo de elemento, el mapeo del id a su código entero, y rellenar las columnas numéricas.

In [ ]:
# Columnas one-hot de cada tipo
cols_distrito_oh = [c for c in features if c.startswith('distrito_')]
cols_franja_oh   = [c for c in features if c.startswith('franja_')]
cols_tipoelem_oh = [c for c in features if c.startswith('tipo_elem_')]

# Invertimos el mapeo original: id real del sensor -> código entero usado por el modelo
codigo_por_id = {id_orig: codigo for codigo, id_orig in mapeo_sensores.items()}


Replicamos la función `franja_horaria` del notebook 02 para que la predicción use exactamente la misma agrupación.

In [ ]:
def _franja(h: int) -> str:
    if   h < 6:  return 'madrugada'
    elif h < 12: return 'mañana'
    elif h < 15: return 'mediodia'
    elif h < 21: return 'tarde'
    else:        return 'noche'


Y ahora la función de predicción en sí. Va comentada paso a paso porque cada bloque tiene su lógica.

In [ ]:
def predict_congestion(id_sensor: int, hora: int, dia_semana: int, mes: int = 3):
    '''Predice si habrá congestión en un sensor a una hora y día de la semana dados.'''

    # 1. Buscamos el sensor en el catálogo para obtener distrito, coordenadas, nombre
    sensor = df_sensores[df_sensores['id'] == id_sensor]
    if sensor.empty:
        raise ValueError(f'Sensor {id_sensor} no está en el catálogo')
    sensor = sensor.iloc[0]

    # 2. Construimos las features derivadas del momento (como en el notebook 02)
    es_laborable  = int(dia_semana < 5)
    es_hora_punta = int(es_laborable and (7 <= hora <= 9 or 17 <= hora <= 20))

    # 3. Creamos una fila con el esqueleto correcto: todas las features a 0
    fila = pd.Series(0, index=features, dtype=float)

    # 4. Rellenamos las numéricas
    fila['hora']          = hora
    fila['dia_semana']    = dia_semana
    fila['mes']           = mes
    fila['es_laborable']  = es_laborable
    fila['es_hora_punta'] = es_hora_punta
    fila['utm_x']         = sensor['utm_x']
    fila['utm_y']         = sensor['utm_y']
    fila['id_sensor']     = codigo_por_id.get(id_sensor, -1)

    # 5. One-hot: activamos la columna correspondiente a cada categoría
    col_d = f"distrito_{sensor['distrito']}"
    col_f = f"franja_{_franja(hora)}"
    col_t = f"tipo_elem_{sensor['tipo_elem']}"
    for c in (col_d, col_f, col_t):
        if c in fila.index:
            fila[c] = 1

    # 6. Llamamos al modelo. Necesita un DataFrame 2D, no una Series.
    X = fila.to_frame().T
    prob = modelo.predict_proba(X)[0, 1]
    etiqueta = 'CONGESTIONADO' if prob >= 0.5 else 'FLUIDO'

    return {
        'estado':       etiqueta,
        'probabilidad': prob,
        'sensor':       sensor['nombre'],
        'distrito':     sensor['distrito'],
    }


### Escenarios de prueba

Vamos a probar la función con varios escenarios, cada uno elegido para cubrir una situación distinta. Usamos sensores aleatorios del catálogo (con semilla fija para que los ejemplos sean reproducibles).

In [ ]:
dias = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']

# Seleccionamos 5 sensores del catálogo con semilla fija
ids_demo = df_sensores['id'].sample(n=5, random_state=7).tolist()

escenarios = [
    (ids_demo[0], 8,  0),   # Lunes a las 8h — hora punta laborable
    (ids_demo[1], 14, 2),   # Miércoles al mediodía
    (ids_demo[2], 3,  5),   # Sábado de madrugada
    (ids_demo[3], 19, 4),   # Viernes por la tarde — punta de tarde
    (ids_demo[4], 11, 6),   # Domingo por la mañana
]

filas = []
for id_sensor, hora, dow in escenarios:
    r = predict_congestion(id_sensor, hora, dow)
    filas.append({
        'Sensor':       r['sensor'][:40],
        'Distrito':     r['distrito'],
        'Día':          dias[dow],
        'Hora':         f'{hora:02d}:00',
        'Predicción':   r['estado'],
        'Probabilidad': f'{r["probabilidad"]:.0%}',
    })

pd.DataFrame(filas)


Las predicciones responden razonablemente a la intuición: más probabilidad en laborables en hora punta, menos en madrugada o domingo. Si alguna predicción se sale de la intuición (por ejemplo, un sensor periférico que marca alta probabilidad un domingo) eso suele significar que ese punto concreto tiene un patrón local que el modelo ha aprendido y el sentido común general no alcanzaba a intuir. Es una de las ventajas de un modelo con acceso al identificador del sensor: aprende idiosincrasias locales.

### Variación intradía de un sensor concreto

Para cerrar la sección, una visualización que muestra el poder de la función: la probabilidad de congestión a lo largo de todo un día, para el mismo sensor, comparando laborable y domingo. Es la vista que pondríamos en una interfaz real.

In [ ]:
id_visualizacion = ids_demo[0]
sensor_viz = df_sensores[df_sensores['id'] == id_visualizacion].iloc[0]

horas = list(range(24))
prob_laborable = [predict_congestion(id_visualizacion, h, 2)['probabilidad'] for h in horas]
prob_domingo   = [predict_congestion(id_visualizacion, h, 6)['probabilidad'] for h in horas]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horas, prob_laborable, marker='o', color=COLOR_PRIMARIO,
        linewidth=1.8, label='Miércoles (laborable)')
ax.plot(horas, prob_domingo, marker='o', color=COLOR_SECUNDARIO,
        linewidth=1.8, label='Domingo')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1,
           label='Umbral de decisión')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Probabilidad de congestión')
ax.set_title(f'Perfil diario de probabilidad — {sensor_viz["nombre"][:50]}')
ax.set_xticks(range(0, 24, 2))
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


La gráfica deja ver perfectamente las dos cosas que el modelo ha aprendido. Primero, la forma del perfil de tráfico con sus dos puntas en laborables. Segundo, que esa forma cambia radicalmente los domingos, sin puntas claras y con probabilidad mucho más baja todo el día. El modelo ha capturado sin problemas la interacción hora × día de la semana que en el EDA anticipábamos.

## 8. Conclusiones del proyecto

Hemos construido un pipeline de principio a fin: descarga de los datos abiertos del Ayuntamiento, análisis exploratorio de los patrones temporales y espaciales, limpieza de los errores de sensor, feature engineering, comparativa de tres modelos con validación cruzada y evaluación final sobre un conjunto de test intacto. Las decisiones metodológicas clave fueron justificadas con los datos: el umbral de congestión del 70% lo fijamos a partir de la CDF de la variable `carga`; la métrica principal fue F1 y no accuracy por el desbalance de clases; evitamos data leakage dejando fuera las variables que se miden en el mismo instante que la carga.

**Qué funciona.** El modelo final separa con bastante claridad entre tráfico fluido y congestionado, con AUC alto y métricas en test que son coherentes con las de validación cruzada — señal de que generaliza bien. Las variables que más pesan son las que el análisis exploratorio ya señalaba como informativas: la hora del día y el sensor concreto. Es decir, el modelo no ha aprendido ninguna combinación rara de variables; ha aprendido lo que se espera que aprenda.

**Qué no funciona tan bien.** El modelo falla más en las horas de transición y en algunos distritos céntricos. Parte de esos fallos se deben a que la frontera entre fluido y congestionado en esos contextos depende de factores que no estamos midiendo — clima, incidentes puntuales, eventos deportivos. El análisis de falsos negativos revela que muchos de los atascos que se nos escapan son casos en los que el modelo dudaba. Con un ajuste del umbral se pueden recuperar a costa de más falsas alarmas, dependiendo del balance que prefiera el usuario final.

**Qué aporta.** Un sistema que, dado un sensor y un momento, devuelve una predicción de congestión en milisegundos. La función de demo lo demuestra de forma tangible. Para un ayuntamiento o un servicio de navegación sería directamente integrable — con los datos de Madrid, que son abiertos, esto es una aplicación real y no un ejercicio académico.

## 9. Posibles mejoras

El proyecto se puede hacer crecer en varias direcciones naturales, en orden creciente de esfuerzo.

**Más datos históricos.** Cargar un año completo en lugar de un mes permitiría al modelo aprender patrones estacionales — el tráfico de agosto, los puentes de mayo, la operación salida — que con datos de un solo mes no pueden distinguirse de simple ruido. Es la mejora con mejor ratio esfuerzo/resultado.

**Datos meteorológicos.** La lluvia desplaza las horas punta y aumenta la carga general. AEMET publica datos históricos abiertos que se podrían cruzar por fecha. Seguramente explicaría una parte de los errores de transición que vimos.

**Memoria temporal.** Añadir la carga del propio sensor en la hora anterior como feature convertiría esto en un modelo de serie temporal. El salto de calidad sería grande pero también cambia el enfoque: ya no sería "dame un momento y te digo cómo estará" sino "dame un momento y su contexto reciente y te digo cómo estará".

**Ajuste fino del umbral.** En la sección de errores vimos que hay una masa de falsos negativos en probabilidades cercanas al umbral actual. Bajar ese umbral recuperaría muchos de esos casos. La decisión exacta depende del caso de uso y se podría dejar como parámetro configurable.

**Integración con la API en tiempo real.** El Ayuntamiento ofrece una API que da las medidas del minuto actual. Conectarla con el modelo convertiría el sistema en un monitor en vivo: pintar en un mapa las zonas de probabilidad alta de atasco en el momento presente y para las próximas horas. Es la extensión más llamativa pero también la que más infraestructura requiere, y queda fuera del alcance de este trabajo de 3 créditos.

Cada una de estas mejoras se puede abordar sin tocar la estructura del proyecto: el dataset procesado es el punto de enganche, y tanto el notebook 03 como el 04 funcionarían sin cambios con un dataset más rico.